In [ ]:
<a href="https://colab.research.google.com" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### ArcFace + YOLOv8n + Multi-Detector Fallback + L2-Normalized Matching

| Step | Description |
|------|-------------|
| 1 | ✅ Verify GPU |
| 2 | 📦 Install packages |
| 3 | 📂 Mount Google Drive |
| 4 | ⚙️ Configuration |
| 5 | 🔧 Helper functions |
| 6 | 🔧 Preprocess & cache to SSD |
| 7 | 🔍 Diagnose all detectors |
| 8 | 🧠 Build Yifer embeddings |
| 9 | 🔍 Debug distances |
| 10 | 🔍 Classify group photos |
| 11 | 📁 Save results to Drive |
| 12 | 📊 Charts & final report |

> **⚠️ Before running:** `Runtime → Change runtime type → GPU (T4)`

---
## ✅ Step 1: Verify GPU

In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if r.returncode == 0:
    print('🟢 GPU DETECTED!\n'); print(r.stdout[:600])
else:
    print('🔴 No GPU! → Runtime → Change runtime type → GPU (T4)')
    raise SystemExit('Enable GPU first!')
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f'\n✅ TensorFlow sees {len(gpus)} GPU(s)')

---
## 📦 Step 2: Install Libraries

In [ ]:
%%capture
!pip install deepface tf-keras retina-face mtcnn tqdm scikit-learn ultralytics -q

In [ ]:
import os, shutil, warnings, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageOps
from tqdm.notebook import tqdm
from scipy.spatial.distance import cosine as cosine_dist
# FIX: Must be set BEFORE deepface import to fix RetinaFace KerasTensor bug
os.environ['TF_USE_LEGACY_KERAS'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
from deepface import DeepFace
warnings.filterwarnings('ignore')
print('✅ All imports done!')

---
## 📂 Step 3: Mount Google Drive
> Upload to Drive first:
> - `MyDrive/practice/yife/` ← your photos of Yifer
> - `MyDrive/practice/group_photos/` ← group photos to classify

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive mounted!')

---
## ⚙️ Step 4: Configuration

In [ ]:
# ===== EDIT THESE IF NEEDED =====
BASE_DIR    = '/content/drive/MyDrive/practice'
YIFER_DIR   = os.path.join(BASE_DIR, 'yife')
GROUP_DIR   = os.path.join(BASE_DIR, 'group_photos')
CLASS1_DIR  = os.path.join(BASE_DIR, 'class_1')
CLASS2_DIR  = os.path.join(BASE_DIR, 'class_2')
PREPROC_DIR = '/content/preprocessed'  # local SSD = fast

MODEL_NAME = 'ArcFace'

# ArcFace cosine threshold: lower = stricter. Default=0.68
THRESHOLD = 0.65

MAX_DIM = 1024
SUPPORTED_EXTS = ('.jpg','.jpeg','.png','.bmp','.webp')

# FIX: 'yolov8n' is the correct name (not 'yolov8')
DETECTOR_FALLBACK_ORDER = ['yolov8n','retinaface','mtcnn','opencv','ssd']

for d in [CLASS1_DIR, CLASS2_DIR, PREPROC_DIR,
          f'{PREPROC_DIR}/yifer', f'{PREPROC_DIR}/group']:
    os.makedirs(d, exist_ok=True)

print('✅ Config ready!')
for lbl, p in [('yife', YIFER_DIR), ('group_photos', GROUP_DIR)]:
    ok = os.path.isdir(p)
    n  = len([f for f in os.listdir(p) if f.lower().endswith(SUPPORTED_EXTS)]) if ok else 0
    print(f'   {lbl:<15}: {"OK - "+str(n)+" photos" if ok else "NOT FOUND"}')
print(f'\n   Model     : {MODEL_NAME}')
print(f'   Threshold : {THRESHOLD}  (ArcFace default=0.68)')
print(f'   Detectors : {DETECTOR_FALLBACK_ORDER}')

---
## 🔧 Step 5: Helper Functions

In [ ]:
def preprocess_image(path, max_dim=MAX_DIM):
    try:
        img = Image.open(path)
        img = ImageOps.exif_transpose(img).convert('RGB')
        w, h = img.size
        if max(w, h) > max_dim:
            s = max_dim / max(w, h)
            img = img.resize((int(w*s), int(h*s)), Image.LANCZOS)
        return img
    except:
        return None

def get_files(folder):
    return sorted([f for f in os.listdir(folder)
                   if f.lower().endswith(SUPPORTED_EXTS)])

def save_img(pil_img, path):
    pil_img.save(path, 'JPEG', quality=92, optimize=True)

def l2_normalize(v):
    norm = np.linalg.norm(v)
    return v / (norm + 1e-10)

def show_faces(img_path, detector, title='Detection'):
    """Show detected faces. Bounding boxes drawn on native pixel coords."""
    try:
        faces = DeepFace.extract_faces(
            img_path=img_path, detector_backend=detector,
            enforce_detection=False, align=False)
        img = np.array(Image.open(img_path).convert('RGB'))
        fig, ax = plt.subplots(figsize=(8, 6))
        fig.patch.set_facecolor('#0f0f23')
        ax.set_facecolor('#0f0f23')
        ax.imshow(img)
        cnt = 0
        for i, fd in enumerate(faces):
            r = fd.get('facial_area', {})
            c = fd.get('confidence', 0)
            if r and r.get('w', 0) > 10:
                cnt += 1
                rect = patches.Rectangle(
                    (r['x'], r['y']), r['w'], r['h'],
                    linewidth=3, edgecolor='#00ff88', facecolor='none')
                ax.add_patch(rect)
                ax.text(r['x'], r['y'] - 8, f'Face {i+1} ({c:.2f})',
                    color='white', fontsize=10,
                    bbox=dict(boxstyle='round', facecolor='#00ff88', alpha=0.85))
        ax.set_title(f'{title}  [{detector}]  {cnt} face(s)',
            color='white', fontsize=12, fontweight='bold')
        ax.axis('off')
        plt.tight_layout(); plt.show()
        return cnt
    except Exception as e:
        print(f'show_faces error: {e}'); return 0

print('✅ Helper functions ready!')

---
## 🔧 Step 6: Preprocess & Cache Photos to Local SSD
> Drive → local SSD = **10-20x faster** processing. Cache reused on re-run.

In [ ]:
def preprocess_folder(src_dir, dst_dir, label):
    files = get_files(src_dir)
    valid = []; errors = 0
    for fn in tqdm(files, desc=f'Preprocessing {label}'):
        dest = os.path.join(dst_dir, os.path.splitext(fn)[0] + '.jpg')
        if not os.path.exists(dest):
            img = preprocess_image(os.path.join(src_dir, fn))
            if img:
                save_img(img, dest)
            else:
                errors += 1; continue
        valid.append((fn, dest))
    print(f'  {label}: {len(valid)} ready | {errors} errors')
    return valid

print('=' * 55)
print('🔧 PREPROCESSING → Local SSD')
print('=' * 55)
yifer_all         = preprocess_folder(YIFER_DIR, f'{PREPROC_DIR}/yifer', 'Yifer')
yifer_valid_paths = [dest for _, dest in yifer_all]
group_valid_paths = preprocess_folder(GROUP_DIR, f'{PREPROC_DIR}/group', 'Group')
print(f'\n📊 Yifer: {len(yifer_valid_paths)} | Group: {len(group_valid_paths)}')

---
## 🔍 Step 7: Diagnose — Test All Detectors on Yifer Photos
> Auto-selects the best detector from your environment.
> **Fixes applied:**
>   - `yolov8n` is the correct backend name (not `yolov8`)
>   - `TF_USE_LEGACY_KERAS=1` (set in Step 2) fixes RetinaFace
>   - mediapipe removed (broken in newer Colab versions)

In [ ]:
print('=' * 55)
print('🔍 DETECTOR DIAGNOSIS')
print('=' * 55)

test_photos = yifer_valid_paths[:3]
# mediapipe removed - broken in recent Colab; yolov8n is the correct name
ALL_DET = ['yolov8n','retinaface','mtcnn','opencv','ssd','yunet','centerface']
det_scores = {}

for tp in test_photos:
    print(f'\n📸 {os.path.basename(tp)}')
    for det in ALL_DET:
        try:
            faces = DeepFace.extract_faces(
                img_path=tp, detector_backend=det,
                enforce_detection=False, align=False)
            n = len([f for f in faces if f.get('confidence', 1) > 0.3])
            det_scores[det] = det_scores.get(det, 0) + n
            status = f'✅ {n} face(s)' if n > 0 else '⚠️  0 faces'
            print(f'   {det:<15}: {status}')
        except Exception as e:
            print(f'   {det:<15}: ❌ {str(e)[:60]}')

if det_scores:
    best = max(det_scores, key=det_scores.get)
    DETECTOR_FALLBACK_ORDER = [best] + [
        d for d in ['yolov8n','retinaface','mtcnn','opencv','ssd']
        if d != best]
    LOCKED_DETECTOR = best
    print(f'\n🏆 Best detector: [{best}]  (scores: {det_scores})')
    print(f'   Fallback order: {DETECTOR_FALLBACK_ORDER}')
else:
    LOCKED_DETECTOR = 'mtcnn'
    DETECTOR_FALLBACK_ORDER = ['mtcnn','opencv','ssd']
    print('⚠️ All failed. Defaulting to mtcnn.')

print(f'\n👁️ Visual check with [{LOCKED_DETECTOR}]:')
show_faces(test_photos[0], LOCKED_DETECTOR, title='Best Detector')

---
## 🧠 Step 8: Build Yifer Embeddings (Multi-Detector Fallback)
> Extracts ArcFace embeddings, L2-normalizes them,
> and computes a **mean centroid** for robust matching.

In [ ]:
def get_embeddings_fallback(img_path):
    """Try each detector in fallback order. Return first success."""
    for det in DETECTOR_FALLBACK_ORDER:
        try:
            result = DeepFace.represent(
                img_path=img_path, model_name=MODEL_NAME,
                detector_backend=det, enforce_detection=True, align=True)
            if result:
                return result, det
        except:
            continue
    return [], 'none'

print('=' * 55)
print('🧠 BUILDING YIFER EMBEDDINGS')
print('=' * 55)
print(f'Fallback: {DETECTOR_FALLBACK_ORDER}  |  Model: {MODEL_NAME}\n')

yifer_embeddings_raw = []
sample_path = None; sample_det = None
no_face = 0; det_stats = {}

for img_path in tqdm(yifer_valid_paths, desc='Extracting Yifer embeddings'):
    results, used = get_embeddings_fallback(img_path)
    if not results:
        no_face += 1; continue
    for fd in results:
        yifer_embeddings_raw.append(np.array(fd['embedding']))
    det_stats[used] = det_stats.get(used, 0) + 1
    if sample_path is None:
        sample_path = img_path; sample_det = used

print(f'\n✅ Raw embeddings  : {len(yifer_embeddings_raw)}')
print(f'⚠️  No face photos  : {no_face}')
for det, cnt in sorted(det_stats.items(), key=lambda x: -x[1]):
    print(f'   {det:<15}: {cnt} photo(s)')

if len(yifer_embeddings_raw) == 0:
    raise ValueError('❌ No embeddings found! Check yife/ folder.')

yifer_embeddings = [l2_normalize(e) for e in yifer_embeddings_raw]
yifer_mean_emb   = l2_normalize(np.mean(yifer_embeddings, axis=0))

print(f'\n✅ L2-normalized {len(yifer_embeddings)} embeddings')
print(f'✅ Mean centroid computed ({yifer_mean_emb.shape[0]}D)')

if sample_path:
    print(f'\n👁️ Sample detection [{sample_det}]:')
    show_faces(sample_path, sample_det, title=f'Yifer — {sample_det}')

### 📸 Yifer Reference Grid

In [ ]:
paths = yifer_valid_paths[:16]
cols = 4; rows = (len(paths) + 3) // 4
fig, axes = plt.subplots(rows, cols, figsize=(15, rows * 3.8))
fig.patch.set_facecolor('#0f0f23')
axes = np.array(axes).flatten()
for i, p in enumerate(paths):
    img = preprocess_image(p, max_dim=250)
    if img:
        axes[i].imshow(np.array(img))
        axes[i].set_title(f'Ref {i+1}', color='#00ff88', fontsize=9)
    axes[i].axis('off')
for j in range(len(paths), len(axes)):
    axes[j].axis('off')
plt.suptitle(f'🧠 Yifer — {len(yifer_embeddings)} embeddings',
             color='white', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 🔍 Step 9: Debug — Inspect Actual Cosine Distances
> Run BEFORE full classification to verify threshold is set correctly.
> Expected: Yifer faces ~0.30-0.50 | non-Yifer faces ~0.60-0.90

In [ ]:
print('=' * 55)
print('🔍 DISTANCE DEBUG')
print('=' * 55)
print(f'Yifer embeddings : {len(yifer_embeddings)}')
print(f'Threshold        : {THRESHOLD}\n')

for orig_fn, preproc_path in group_valid_paths[:8]:
    print(f'📸 {orig_fn[:50]}')
    try:
        results = DeepFace.represent(
            img_path=preproc_path, model_name=MODEL_NAME,
            detector_backend=LOCKED_DETECTOR,
            enforce_detection=False, align=True)
        if not results:
            print('   ⚠️ No faces detected'); continue
        print(f'   Faces found: {len(results)}')
        for i, fd in enumerate(results):
            emb     = l2_normalize(np.array(fd['embedding']))
            d_mean  = cosine_dist(emb, yifer_mean_emb)
            d_best  = min(cosine_dist(emb, ref) for ref in yifer_embeddings)
            d_final = min(d_mean, d_best)
            match   = '✅ MATCH' if d_final <= THRESHOLD else '❌ NO MATCH'
            print(f'   Face {i+1}: mean={d_mean:.4f}  best={d_best:.4f}  '
                  f'final={d_final:.4f}  → {match}')
    except Exception as e:
        print(f'   Error: {e}')
    print()

print('-' * 55)
print(f'Threshold={THRESHOLD} | Raise if Yifer missed | Lower if too many false positives')

---
## 🔍 Step 10: Classify Group Photos
> Uses the locked detector + full fallback chain.
> Compares each face against both the mean centroid AND individual embeddings.

In [ ]:
def is_yifer_in_image(img_path, threshold=THRESHOLD):
    """Returns (found, best_dist, n_faces, detector_used)."""
    dets = [LOCKED_DETECTOR] + [
        d for d in DETECTOR_FALLBACK_ORDER if d != LOCKED_DETECTOR]
    for det in dets:
        try:
            results = DeepFace.represent(
                img_path=img_path, model_name=MODEL_NAME,
                detector_backend=det, enforce_detection=True, align=True)
            best_dist = 1.0
            for fd in results:
                emb     = l2_normalize(np.array(fd['embedding']))
                d_mean  = cosine_dist(emb, yifer_mean_emb)
                d_best  = min(cosine_dist(emb, ref) for ref in yifer_embeddings)
                d_final = min(d_mean, d_best)
                if d_final < best_dist:
                    best_dist = d_final
                if d_final <= threshold:
                    return True, d_final, len(results), det
            return False, best_dist, len(results), det
        except:
            continue
    return False, 1.0, 0, 'none'

print('✅ Classification function ready!')
print(f'   Locked detector : {LOCKED_DETECTOR}')
print(f'   Threshold       : {THRESHOLD}')
print(f'   Matching        : L2-norm + mean centroid + all individual refs')

print('\n🧪 Sanity test on first 3 group photos:')
for fn, pp in group_valid_paths[:3]:
    found, dist, n, det = is_yifer_in_image(pp)
    tag = '🟢 YIFER' if found else '🔴 OTHER'
    print(f'  {tag}  dist={dist:.4f}  faces={n}  det={det}  {fn[:35]}')

In [ ]:
print('=' * 55)
print('🔍 CLASSIFYING GROUP PHOTOS')
print('=' * 55)
print(f'Photos: {len(group_valid_paths)} | Embeddings: {len(yifer_embeddings)} | Threshold: {THRESHOLD}\n')

class1_files  = []; class2_files = []; results_log = []
no_face_total = 0;  start_time   = time.time()

for orig_fn, preproc_path in tqdm(group_valid_paths, desc='Classifying'):
    found, dist, n_faces, det = is_yifer_in_image(preproc_path)
    if n_faces == 0:
        no_face_total += 1
    results_log.append((orig_fn, 'C1' if found else 'C2', dist, n_faces, det))
    if found:
        class1_files.append(orig_fn)
    else:
        class2_files.append(orig_fn)

elapsed = time.time() - start_time
print(f'\n{"═"*50}')
print(f'  🎯 CLASSIFICATION COMPLETE!')
print(f'{"═"*50}')
print(f'  ⏱️  Time     : {elapsed:.1f}s  ({elapsed/max(len(group_valid_paths),1):.2f}s/img)')
print(f'  🟢 Class 1  : {len(class1_files):>5} photos  (Yifer present)')
print(f'  🔴 Class 2  : {len(class2_files):>5} photos  (Yifer absent)')
print(f'  ⚪ No face  : {no_face_total:>5} photos')
print(f'  📊 Total    : {len(group_valid_paths):>5} photos')
print(f'{"═"*50}')

---
## 📁 Step 11: Save Results to Google Drive

In [ ]:
print('📁 Copying to Google Drive...')
# Auto-create folders if they do not exist yet
os.makedirs(CLASS1_DIR, exist_ok=True)
os.makedirs(CLASS2_DIR, exist_ok=True)

errs = []
print(f'\n🟢 {len(class1_files)} → class_1/')
for fn in tqdm(class1_files, desc='→ class_1'):
    try:
        shutil.copy2(os.path.join(GROUP_DIR, fn), os.path.join(CLASS1_DIR, fn))
    except Exception as e:
        errs.append((fn, str(e)))

print(f'\n🔴 {len(class2_files)} → class_2/')
for fn in tqdm(class2_files, desc='→ class_2'):
    try:
        shutil.copy2(os.path.join(GROUP_DIR, fn), os.path.join(CLASS2_DIR, fn))
    except Exception as e:
        errs.append((fn, str(e)))

print(f'\n✅ class_1: {len(os.listdir(CLASS1_DIR))} | class_2: {len(os.listdir(CLASS2_DIR))}')
if errs:
    print(f'⚠️ {len(errs)} copy error(s):')
    for fn, err in errs[:5]: print(f'  {fn}: {err}')
print('🎉 Done! Check Google Drive → practice/class_1 & class_2')

---
## 📊 Step 12: Visual Results & Final Report

In [ ]:
def show_class_grid(flist, src, title, color, n=12):
    if not flist:
        print(f'No files in {title}'); return
    show = flist[:n]; cols = 4; rows = (len(show) + 3) // 4
    fig, axes = plt.subplots(rows, cols, figsize=(15, rows * 3.8))
    fig.patch.set_facecolor('#0f0f23')
    axes = np.array(axes).flatten()
    for i, fn in enumerate(show):
        img = preprocess_image(os.path.join(src, fn), max_dim=350)
        if img:
            axes[i].imshow(np.array(img))
            axes[i].set_title(fn[:22], fontsize=7, color='white')
        axes[i].axis('off')
    for j in range(len(show), len(axes)):
        axes[j].axis('off')
    plt.suptitle(f'{title}  —  {len(flist)} total',
                 color=color, fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()

show_class_grid(class1_files, GROUP_DIR, '🟢 CLASS 1 — Contains Yifer', '#00ff88')
show_class_grid(class2_files, GROUP_DIR, '🔴 CLASS 2 — No Yifer',       '#ff4757')

In [ ]:
fig = plt.figure(figsize=(16, 5))
fig.patch.set_facecolor('#0f0f23')

ax1 = fig.add_subplot(1, 3, 1); ax1.set_facecolor('#0f0f23')
_, _, autos = ax1.pie(
    [len(class1_files), len(class2_files)],
    labels=[f'Class 1\n(Yifer)\n{len(class1_files)}',
            f'Class 2\n(Others)\n{len(class2_files)}'],
    colors=['#00ff88','#ff4757'], explode=(0.06, 0.06),
    autopct='%1.1f%%', startangle=140,
    textprops={'color':'white','fontsize':11,'fontweight':'bold'},
    wedgeprops={'edgecolor':'#0f0f23','linewidth':3})
for a in autos: a.set_color('white'); a.set_fontsize(12)
ax1.set_title('Split', color='white', fontsize=13, fontweight='bold')

ax2 = fig.add_subplot(1, 3, 2); ax2.set_facecolor('#16213e')
bars = ax2.bar(['Class 1','Class 2','No Face'],
    [len(class1_files), len(class2_files), no_face_total],
    color=['#00ff88','#ff4757','#ffa502'], width=0.5)
for b in bars:
    ax2.text(b.get_x()+b.get_width()/2, b.get_height()+1,
             str(int(b.get_height())), ha='center',
             color='white', fontsize=12, fontweight='bold')
ax2.tick_params(colors='white')
ax2.set_title('Breakdown', color='white', fontsize=13)
for s in ['top','right']:   ax2.spines[s].set_visible(False)
for s in ['bottom','left']: ax2.spines[s].set_color('#444')

ax3 = fig.add_subplot(1, 3, 3); ax3.set_facecolor('#16213e')
d1 = [r[2] for r in results_log if r[1]=='C1']
d2 = [r[2] for r in results_log if r[1]=='C2']
if d1: ax3.hist(d1, bins=25, color='#00ff88', alpha=0.7, label='Class 1')
if d2: ax3.hist(d2, bins=25, color='#ff4757', alpha=0.6, label='Class 2')
ax3.axvline(THRESHOLD, color='yellow', linestyle='--',
            lw=2, label=f'Threshold={THRESHOLD}')
ax3.tick_params(colors='white')
ax3.set_xlabel('Cosine Distance', color='white')
ax3.set_title('Distance Distribution', color='white', fontsize=13)
ax3.legend(fontsize=9, facecolor='#0f0f23', labelcolor='white')
for s in ['top','right']:   ax3.spines[s].set_visible(False)
for s in ['bottom','left']: ax3.spines[s].set_color('#444')

plt.suptitle(f'🎯 Report — {MODEL_NAME} + {LOCKED_DETECTOR}',
             color='white', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/content/report.png', dpi=150,
            bbox_inches='tight', facecolor='#0f0f23')
plt.show()
shutil.copy2('/content/report.png',
             os.path.join(BASE_DIR, 'classification_report.png'))
print('📊 Report saved to Drive!')

In [ ]:
W = 54
print('╔' + '═'*W + '╗')
print('║  🎯  FINAL REPORT' + ' '*(W-14) + '║')
print('╠' + '═'*W + '╣')
def row(k, v): print(f'║  {k:<18}: {str(v):<{W-22}}  ║')
row('Model',      MODEL_NAME)
row('Threshold',  THRESHOLD)
row('Detector',   LOCKED_DETECTOR)
row('GPU',        f'{len(gpus)} device(s)')
print('╠' + '═'*W + '╣')
row('Yifer refs',  len(yifer_valid_paths))
row('Embeddings',  len(yifer_embeddings))
row('Group photos',len(group_valid_paths))
print('╠' + '═'*W + '╣')
row('🟢 Class 1 (Yifer)', len(class1_files))
row('🔴 Class 2 (Other)', len(class2_files))
row('⚪ No face',      no_face_total)
row('⏱️  Time',        f'{elapsed:.1f}s')
print('╚' + '═'*W + '╝')
print('\nSample results (first 20):')
print(f'{"File":<45} {"Cls":<5} {"Dist":<8} {"Faces":<6} Det')
print('-' * 78)
for fn, lbl, dist, nf, det in results_log[:20]:
    print(f'{fn[:43]:<45} {lbl:<5} {dist:.4f}  {nf:<6} {det}')

---
## 🔄 Optional: Re-tune Threshold & Re-copy
> - Yifer still missed → **raise** threshold (e.g. `0.70`)
> - Too many false positives → **lower** threshold (e.g. `0.55`)

In [ ]:
NEW_THRESHOLD = 0.70  # <- change this value
c1, c2 = [], []
for fn, pp in tqdm(group_valid_paths, desc=f'Re-testing @{NEW_THRESHOLD}'):
    found, *_ = is_yifer_in_image(pp, threshold=NEW_THRESHOLD)
    (c1 if found else c2).append(fn)
print(f'\nThreshold={NEW_THRESHOLD}: Class1={len(c1)} | Class2={len(c2)}')
print('Run next cell to apply and re-copy to Drive.')

In [ ]:
# WARNING: Clears existing class folders and re-copies with new threshold
for folder in [CLASS1_DIR, CLASS2_DIR]:
    shutil.rmtree(folder, ignore_errors=True)
    os.makedirs(folder, exist_ok=True)
for fn in tqdm(c1, desc='→ class_1'):
    shutil.copy2(os.path.join(GROUP_DIR, fn), os.path.join(CLASS1_DIR, fn))
for fn in tqdm(c2, desc='→ class_2'):
    shutil.copy2(os.path.join(GROUP_DIR, fn), os.path.join(CLASS2_DIR, fn))
print(f'\n✅ class_1: {len(os.listdir(CLASS1_DIR))} | class_2: {len(os.listdir(CLASS2_DIR))}')